# LSEG Daily Returns fuer Euro500 Portfolio (firm_id Cache)

Ziel dieses Notebooks:
- Daily Returns fuer alle Aktien im `euro500` Portfolio ziehen
- Identifier-Fallback robust halten: alle historischen ISIN/RIC je `firm_id`
- Cache **pro Unternehmen (`firm_id`)** statt pro Quartal
- Finale Tabelle nur fuer Tage, an denen die Aktie im Index ist (exakte Mitgliedschaft)


## 1. Setup und Universe laden

- Pfade/Parameter setzen
- `euro500.parquet` laden
- Datentypen bereinigen und `firm_id` als verpflichtenden Schluessel nutzen


In [1]:
from __future__ import annotations

import hashlib
import json
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import lseg.data as ld

import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")

pd.set_option("display.max_columns", 120)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
CACHE_DATA_DIR = BASE_DIR / "cache"
CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = BASE_DIR / "graphs"

CACHE_DIR = CACHE_DATA_DIR / "daily_returns_cache_by_company"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = CACHE_DATA_DIR / "daily_returns_company_cache_manifest.parquet"
OUTPUT_RETURNS_ALL = DATA_DIR / "daily_returns_company_all.parquet"
OUTPUT_DAILY_RETURNS_EURO500 = DATA_DIR / "euro500_daily_returns.parquet"
OUTPUT_MISSING = DATA_DIR / "daily_returns_missing_companies.parquet"
LEGACY_RETURNS_PATH = DATA_DIR / "rets_daily_isin.parquet"
BAD_IDS_PATH = CACHE_DATA_DIR / "daily_returns_bad_ids.csv"
STEP_ROWS_PATH = CACHE_DATA_DIR / "daily_returns_step_rows.parquet"
STEP_CKPT_PATH = CACHE_DATA_DIR / "daily_returns_step_checkpoint.json"

TARGET_END_DATE = pd.Timestamp("2025-12-31")
BETA_LOOKBACK_MONTHS = 6

EURO500_PATH = DATA_DIR / "euro500.parquet"
if not EURO500_PATH.exists():
    raise FileNotFoundError(f"File not found: {EURO500_PATH}")

df = pd.read_parquet(EURO500_PATH).copy()


## 2. Exakte Mitgliedschaftsfenster je Quartal

Mitgliedschaft pro Zeile gilt von `effective_date` bis einen Tag vor naechster `effective_date`.

from matplotlib.ticker import MaxNLocator


from matplotlib.ticker import MultipleLocator



In [2]:
quarter_calendar = (
    df[["quarter", "effective_date"]]
    .dropna()
    .drop_duplicates()
    .sort_values("effective_date")
    .reset_index(drop=True)
)
quarter_calendar["next_effective_date"] = quarter_calendar["effective_date"].shift(-1)

# Ende inklusiv: Tag vor naechster effective_date, hart gekappt auf TARGET_END_DATE.
max_end = TARGET_END_DATE
quarter_calendar["membership_end_date"] = quarter_calendar["next_effective_date"] - pd.Timedelta(days=1)
quarter_calendar["membership_end_date"] = quarter_calendar["membership_end_date"].fillna(max_end)
quarter_calendar["membership_end_date"] = quarter_calendar["membership_end_date"].clip(upper=max_end)

membership = df.merge(
    quarter_calendar[["quarter", "effective_date", "membership_end_date"]],
    on=["quarter", "effective_date"],
    how="left",
)

# Avoid duplicate date-column names from upstream inputs.
membership = membership.drop(columns=[c for c in ["start_date", "end_date"] if c in membership.columns])
membership["start_date"] = membership["effective_date"]
membership["end_date"] = membership["membership_end_date"]
membership = membership.drop(columns=["membership_end_date"])

membership = membership.dropna(subset=["firm_id", "start_date", "end_date"]).copy()
membership = membership[membership["start_date"] <= membership["end_date"]].copy()

print("Membership rows:", len(membership))
print("Membership companies:", membership["firm_id"].nunique())
print("Date range:", membership["start_date"].min().date(), "to", membership["end_date"].max().date())


Membership rows: 56000
Membership companies: 1245
Date range: 1998-01-01 to 2025-12-31


## 3. Cache- und Pull-Helfer

- Ein Cache-File pro `firm_id`
- Bei vorhandenem Cache nur fehlende Datumssegmente nachladen


In [3]:
@dataclass
class PullResult:
    df: pd.DataFrame
    field_used: str | None
    mode_used: str | None


def _safe_name(firm_id: str) -> str:
    h = hashlib.sha1(firm_id.encode("utf-8")).hexdigest()[:12]
    clean = re.sub(r"[^A-Za-z0-9._-]", "_", firm_id)
    return f"{clean[:80]}__{h}.parquet"


def cache_path_for_company(firm_id: str) -> Path:
    return CACHE_DIR / _safe_name(firm_id)


def _extract_single_series(raw: pd.DataFrame) -> pd.DataFrame:
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["date", "value"])

    w = raw.copy().reset_index()
    if w.empty:
        return pd.DataFrame(columns=["date", "value"])

    def _col_name(c) -> str:
        if isinstance(c, tuple):
            return " ".join([str(x) for x in c if x is not None]).strip().lower()
        return str(c).strip().lower()

    def _is_date_named(c) -> bool:
        n = _col_name(c)
        return ("date" in n) or ("time" in n) or (n == "index")

    # Date must come from a date/time-named column (or first index column fallback),
    # never from numeric value columns.
    preferred = [c for c in w.columns if _is_date_named(c)]
    candidates = preferred if preferred else [w.columns[0]]

    date_col = None
    best_date_non_na = -1
    for c in candidates:
        d = pd.to_datetime(w[c], errors="coerce")
        n = int(d.notna().sum())
        if n > best_date_non_na:
            best_date_non_na = n
            date_col = c

    if date_col is None or best_date_non_na <= 0:
        return pd.DataFrame(columns=["date", "value"])

    w = w.rename(columns={date_col: "date"})
    w["date"] = pd.to_datetime(w["date"], errors="coerce")
    w = w.dropna(subset=["date"]).copy()

    value_col = None
    best_non_na = -1
    for c in w.columns:
        if c == "date":
            continue
        s_num = pd.to_numeric(w[c], errors="coerce")
        n = int(s_num.notna().sum())
        if n > best_non_na:
            best_non_na = n
            value_col = c

    if value_col is None or best_non_na <= 0:
        return pd.DataFrame(columns=["date", "value"])

    out = w[["date", value_col]].copy()
    out = out.rename(columns={value_col: "value"})
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    return out.dropna(subset=["value"]).copy()


def _values_to_returns(s: pd.Series, mode: str) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")

    if mode == "price_level":
        return x.pct_change()

    # mode == return_like: determine percent vs decimal
    abs_q99 = np.nanpercentile(np.abs(x.dropna()), 99) if x.notna().any() else np.nan
    if np.isfinite(abs_q99) and abs_q99 > 1.5:
        return x / 100.0
    return x


def _is_rate_limit_message(msg: str) -> bool:
    m = str(msg).lower()
    return (
        ("too many requests" in m)
        or ("rate limit" in m)
        or ("http 429" in m)
        or ("status 429" in m)
    )


def load_legacy_returns(path: Path) -> pd.DataFrame:
    """
    Normalize existing returns file (rets_daily_isin.parquet) to columns:
    date | pull_id | id_type | ret
    """
    if not path.exists():
        return pd.DataFrame(columns=["date", "pull_id", "id_type", "ret"])

    raw = pd.read_parquet(path).copy()

    date_col = "date" if "date" in raw.columns else None
    if date_col is None:
        for c in raw.columns:
            if "date" in str(c).lower():
                date_col = c
                break
    if date_col is None:
        return pd.DataFrame(columns=["date", "pull_id", "id_type", "ret"])

    id_col = None
    for c in ["ISIN", "isin", "Isin"]:
        if c in raw.columns:
            id_col = c
            break
    if id_col is None:
        return pd.DataFrame(columns=["date", "pull_id", "id_type", "ret"])

    if "ret" in raw.columns:
        ret = pd.to_numeric(raw["ret"], errors="coerce")
    elif "value" in raw.columns:
        v = pd.to_numeric(raw["value"], errors="coerce")
        abs_q99 = np.nanpercentile(np.abs(v.dropna()), 99) if v.notna().any() else np.nan
        ret = v / 100.0 if np.isfinite(abs_q99) and abs_q99 > 1.5 else v
    else:
        return pd.DataFrame(columns=["date", "pull_id", "id_type", "ret"])

    out = pd.DataFrame(
        {
            "date": pd.to_datetime(raw[date_col], errors="coerce"),
            "pull_id": raw[id_col].astype("string").str.strip(),
            "id_type": "ISIN",
            "ret": ret,
        }
    )
    out.loc[out["pull_id"] == "", "pull_id"] = pd.NA
    out = out.dropna(subset=["date", "pull_id", "ret"]).copy()
    out = out.sort_values(["pull_id", "date"]).drop_duplicates(["pull_id", "date"], keep="last")
    return out.reset_index(drop=True)


def pull_one_company_returns(
    pull_id: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    id_type: str | None = None,
    max_retries: int = 4,
    base_sleep: float = 0.7,
    verbose: bool = False,
) -> PullResult:
    id_type = (id_type or "").upper()
    if id_type == "ISIN":
        plans = [
            ("TR.TotalReturn", "return_like"),
            ("TR.PriceClose", "price_level"),
            ("TRDPRC_1", "price_level"),
        ]
    else:
        plans = [
            ("TR.TotalReturn", "return_like"),
            ("PCTCHNG", "return_like"),
            ("TR.PriceClose", "price_level"),
            ("TRDPRC_1", "price_level"),
        ]

    for field, mode in plans:
        last_err = None
        for r in range(max_retries):
            try:
                raw = ld.get_history(
                    universe=[pull_id],
                    fields=[field],
                    start=start.strftime("%Y-%m-%d"),
                    end=end.strftime("%Y-%m-%d"),
                    interval="daily",
                )
                if verbose:
                    try:
                        print(f"[DEBUG raw] id={pull_id} field={field} shape={getattr(raw, 'shape', None)} cols={list(getattr(raw, 'columns', []))[:8]}")
                    except Exception:
                        pass

                ser = _extract_single_series(raw)
                if verbose:
                    if ser.empty:
                        print(f"[DEBUG ser empty] id={pull_id} field={field}")
                    else:
                        print(f"[DEBUG ser] id={pull_id} field={field} rows={len(ser)} range={ser['date'].min()}:{ser['date'].max()}")

                if ser.empty:
                    break

                ser = ser.sort_values("date")
                ser["ret"] = _values_to_returns(ser["value"], mode=mode)
                ser = ser.dropna(subset=["ret"])[["date", "ret"]].copy()
                if verbose:
                    if ser.empty:
                        print(f"[DEBUG ret empty] id={pull_id} field={field} mode={mode}")
                    else:
                        print(f"[DEBUG ret] id={pull_id} field={field} mode={mode} rows={len(ser)} range={ser['date'].min()}:{ser['date'].max()}")

                if not ser.empty:
                    return PullResult(df=ser, field_used=field, mode_used=mode)
                break
            except Exception as e:
                last_err = e
                if verbose:
                    print(f"[DEBUG exception] id={pull_id} field={field} retry={r+1}/{max_retries} err={e}")
                sleep_s = base_sleep * (2 ** r) + random.random() * 0.3
                time.sleep(sleep_s)

        if last_err is not None:
            if _is_rate_limit_message(str(last_err)):
                # Do not silently downgrade to no-data on 429/rate-limit.
                raise RuntimeError(f"RATE_LIMIT: {last_err}")
            if verbose:
                print(f"[WARN] {pull_id} failed on field {field}: {last_err}")

    return PullResult(df=pd.DataFrame(columns=["date", "ret"]), field_used=None, mode_used=None)


def _load_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["date", "ret"])
    d = pd.read_parquet(path)
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d["ret"] = pd.to_numeric(d["ret"], errors="coerce")
    return d.dropna(subset=["date", "ret"]).sort_values("date").copy()


def _save_cache(path: Path, d: pd.DataFrame) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    d.sort_values("date").drop_duplicates(subset=["date"], keep="last").to_parquet(tmp, index=False)
    tmp.replace(path)


def _normalize_seed(seed_returns: pd.DataFrame | None, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    if seed_returns is None or seed_returns.empty:
        return pd.DataFrame(columns=["date", "ret"])

    x = seed_returns.copy()
    x["date"] = pd.to_datetime(x["date"], errors="coerce")
    x["ret"] = pd.to_numeric(x["ret"], errors="coerce")
    x = x.dropna(subset=["date", "ret"]).copy()
    x = x[(x["date"] >= start) & (x["date"] <= end)].copy()
    x = x.sort_values("date").drop_duplicates(subset=["date"], keep="last")
    return x[["date", "ret"]]


def update_company_cache(
    firm_id: str,
    pull_id: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    id_type: str | None = None,
    force_refresh: bool = False,
    seed_returns: pd.DataFrame | None = None,
    verbose: bool = False,
) -> tuple[pd.DataFrame, str | None, str | None]:
    path = cache_path_for_company(firm_id)
    cached = pd.DataFrame(columns=["date", "ret"]) if force_refresh else _load_cache(path)

    seed = _normalize_seed(seed_returns, start=start, end=end)
    if not seed.empty:
        frames = [x for x in [cached, seed] if not x.empty]
        if frames:
            cached = pd.concat(frames, ignore_index=True)
        else:
            cached = pd.DataFrame(columns=["date", "ret"])
        cached = cached.sort_values("date").drop_duplicates(subset=["date"], keep="last")

    segments: list[tuple[pd.Timestamp, pd.Timestamp]] = []
    if cached.empty:
        segments.append((start, end))
    else:
        cmin, cmax = cached["date"].min(), cached["date"].max()
        if start < cmin:
            segments.append((start, cmin - pd.Timedelta(days=1)))
        if end > cmax:
            segments.append((cmax + pd.Timedelta(days=1), end))

    pulled_parts = []
    field_used = None
    mode_used = None
    for s, e in segments:
        if s > e:
            continue
        res = pull_one_company_returns(
            pull_id=pull_id,
            start=s,
            end=e,
            id_type=id_type,
            verbose=verbose,
        )
        if not res.df.empty:
            pulled_parts.append(res.df)
        field_used = field_used or res.field_used
        mode_used = mode_used or res.mode_used

    if pulled_parts:
        frames = [x for x in [cached] + pulled_parts if not x.empty]
        if frames:
            all_df = pd.concat(frames, ignore_index=True)
        else:
            all_df = pd.DataFrame(columns=["date", "ret"])
    else:
        all_df = cached.copy()

    all_df = all_df.dropna(subset=["date", "ret"]).sort_values("date")
    all_df = all_df.drop_duplicates(subset=["date"], keep="last")

    if not all_df.empty or force_refresh:
        _save_cache(path, all_df)

    return all_df, field_used, mode_used







## 4. Firmenweise Pull-Map und Fallback-Kandidaten aufbauen

- Pro `firm_id` Pull-Zeitraum bestimmen (inkl. Beta-Lookback)
- Historische Identifier-Liste je Firma aufbauen (ISIN -> RIC_current -> RIC)


In [4]:
company_base = (
    membership[["firm_id"]]
    .dropna(subset=["firm_id"])
    .drop_duplicates(subset=["firm_id"])
    .reset_index(drop=True)
)

# Pull-Zeitraum aus Mitgliedschaftsfenstern je Unternehmen
span = (
    membership.groupby("firm_id", as_index=False)
    .agg(start_date=("start_date", "min"), end_date=("end_date", "max"))
)
span["start_date"] = span["start_date"] - pd.DateOffset(months=BETA_LOOKBACK_MONTHS)
company_pull_map = company_base.merge(span, on="firm_id", how="left")

# Unternehmens-Metadaten fuer bessere Missing-Analyse
meta_cols = [
    c for c in ["firm_id", "name", "ISIN", "RIC", "RIC_current", "hq_country", "hq_code"]
    if c in membership.columns
]
company_meta = (
    membership[meta_cols]
    .dropna(subset=["firm_id"])
    .drop_duplicates(subset=["firm_id"], keep="first")
    .rename(columns={"name": "company_name"})
)
company_pull_map = company_pull_map.merge(company_meta, on="firm_id", how="left")


def _norm_isin(value) -> str:
    if pd.isna(value):
        return ""
    v = str(value).strip()
    if not v:
        return ""
    # Remove repeated ISIN prefixes (e.g., ISIN:ISIN:ES...) and keep the raw code.
    while v.upper().startswith("ISIN:"):
        v = v.split(":", 1)[1].strip()
    return v


def _build_company_candidates(company_req: pd.DataFrame) -> list[tuple[str, str]]:
    """Collect all unique identifiers for one company across time.

    Order (like LSEG_DataPull): all ISINs -> all RIC_current -> all RIC.
    """
    q = company_req.copy().sort_values("start_date", na_position="last")

    out: list[tuple[str, str]] = []
    seen: set[tuple[str, str]] = set()

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type, v)
        if key in seen:
            return
        seen.add(key)
        out.append(key)

    # 1) all ISINs observed over time
    if "ISIN" in q.columns:
        for val in q["ISIN"]:
            norm = _norm_isin(val)
            if norm:
                _add("ISIN", norm)
                
    # 2) all current RICs observed over time
    if "RIC_current" in q.columns:
        for val in q["RIC_current"]:
            _add("RIC", val)

    # 3) all legacy/raw RICs observed over time
    if "RIC" in q.columns:
        for val in q["RIC"]:
            _add("RIC", val)

    # Include existing pull_id/id_type history too if present, without creating new pull_id columns.
    if "pull_id" in q.columns and "id_type" in q.columns:
        for id_type, pull_id in zip(q["id_type"], q["pull_id"]):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ""
            if it == "ISIN":
                norm = _norm_isin(pull_id)
                if norm:
                    _add("ISIN", norm)
            elif it == "RIC":
                _add("RIC", pull_id)

    return out


company_candidates_map = {}
for ck, g in membership.groupby("firm_id", sort=False):
    if pd.isna(ck):
        continue
    company_candidates_map[str(ck)] = _build_company_candidates(g)

company_pull_map["id_candidates"] = company_pull_map["firm_id"].astype("string").map(
    lambda ck: company_candidates_map.get(str(ck), [])
)
company_pull_map["n_id_candidates"] = company_pull_map["id_candidates"].apply(len)

legacy_returns = load_legacy_returns(LEGACY_RETURNS_PATH)
legacy_by_id = {}
if not legacy_returns.empty:
    for (id_type, pull_id), g in legacy_returns.groupby(["id_type", "pull_id"], dropna=False):
        legacy_by_id[(id_type, pull_id)] = g[["date", "ret"]].copy()

print("Companies to process:", len(company_pull_map))
print("Legacy rows loaded:", len(legacy_returns))
print("Legacy ids loaded  :", len(legacy_by_id))
print("ID candidates stats:", company_pull_map["n_id_candidates"].describe(), sep="\n")
print(company_pull_map[["firm_id", "firm_id", "n_id_candidates"]].head(3))



Companies to process: 1245
Legacy rows loaded: 0
Legacy ids loaded  : 0
ID candidates stats:
count    1245.000000
mean        2.616064
std         0.862062
min         2.000000
25%         2.000000
50%         2.000000
75%         3.000000
max         8.000000
Name: n_id_candidates, dtype: float64
       firm_id      firm_id  n_id_candidates
0  FIRM0000557  FIRM0000557                2
1  FIRM0000171  FIRM0000171                2
2  FIRM0000351  FIRM0000351                4


## 5. Cache-Precheck, Pull und Outputs schreiben

- Cache/Seed-Abdeckung je `firm_id` pruefen
- Nur fehlende Segmente aus LSEG nachladen (Fallback ueber Kandidaten)
- Outputs schreiben: Manifest, `daily_returns_company_all`, `daily_returns_missing_companies` (optional)


In [5]:
# Step 5 — Daily Returns Pull via Standard Series Puller (lean version)

from standard_lseg_series_puller import DailyReturnsPullConfig, run_daily_returns_standard_puller

force_refresh = False
SKIP_LSEG_PULL = False
CHECKPOINT_EVERY_N_PULLS = 25
PULL_ABORT_ON_RATE_LIMIT = True
SKIP_KNOWN_BAD_IDS = True

cfg = DailyReturnsPullConfig(
    target_end_date=TARGET_END_DATE,
    force_refresh=force_refresh,
    skip_lseg_pull=SKIP_LSEG_PULL,
    checkpoint_every_n_pulls=CHECKPOINT_EVERY_N_PULLS,
    skip_known_bad_ids=SKIP_KNOWN_BAD_IDS,
    pull_abort_on_rate_limit=PULL_ABORT_ON_RATE_LIMIT,
)

stats = run_daily_returns_standard_puller(
    company_pull_map=company_pull_map,
    legacy_by_id=legacy_by_id,
    cache_dir=CACHE_DIR,
    manifest_path=MANIFEST_PATH,
    output_returns_all=OUTPUT_RETURNS_ALL,
    output_missing=OUTPUT_MISSING,
    bad_ids_path=BAD_IDS_PATH,
    step_rows_path=STEP_ROWS_PATH,
    step_ckpt_path=STEP_CKPT_PATH,
    config=cfg,
)

print('Standard pull stats:', stats)

# Backward-compatible in-memory variable for later notebook cells
if OUTPUT_MISSING.exists():
    missing_df = pd.read_parquet(OUTPUT_MISSING).copy()
else:
    missing_df = pd.DataFrame(columns=["firm_id", "company_name", "ISIN", "RIC", "RIC_current"])



- total: 1245 | cache_files: 1228
- known_bad_ids: 0
[Pull firm 1/1245] firm_id=FIRM0000557 | cand_used=1/2 | bad_id_skip=0 | coverage=97.31% | index_range=1997-07-01:2025-12-31 | pulled_range=1997-07-01:2026-01-05 | tried_ids: ISIN:DE0005557508
[Pull firm 2/1245] firm_id=FIRM0000171 | cand_used=1/2 | bad_id_skip=0 | coverage=97.24% | index_range=1997-07-01:2025-12-31 | pulled_range=1997-07-01:2026-01-05 | tried_ids: ISIN:IT0003132476
[Pull firm 3/1245] firm_id=FIRM0000351 | cand_used=1/4 | bad_id_skip=0 | coverage=97.24% | index_range=1997-07-01:2025-12-31 | pulled_range=1997-07-01:2026-01-05 | tried_ids: ISIN:IT0003497168
[Pull firm 4/1245] firm_id=FIRM0002482 | cand_used=1/2 | bad_id_skip=0 | coverage=97.36% | index_range=1997-07-01:2025-12-31 | pulled_range=1997-07-01:2026-01-05 | tried_ids: ISIN:ES0113211835
[Pull firm 5/1245] firm_id=FIRM0000208 | cand_used=1/3 | bad_id_skip=0 | coverage=96.87% | index_range=1997-07-01:2025-12-31 | pulled_range=1997-10-21:2026-01-05 | tried_ids: 

KeyboardInterrupt: 

## 6. Exakt auf Index-Mitgliedschaft filtern

`returns_in_index` enthaelt nur Tage, an denen die Aktie im Portfolio/Index enthalten war.


In [ ]:
# Expand membership windows to business days for exact day-level filter
m_cols = [
    c for c in ["quarter", "start_date", "end_date", "firm_id", "name", "ISIN", "RIC", "RIC_current"]
    if c in membership.columns
]
m = membership[m_cols].copy()

m = m.dropna(subset=["firm_id", "start_date", "end_date"]).copy()

parts = []
for _, r in m.iterrows():
    dates = pd.bdate_range(r["start_date"], r["end_date"], freq="B")
    if len(dates) == 0:
        continue
    part = pd.DataFrame({"date": dates})
    for c in m_cols:
        if c in {"start_date", "end_date"}:
            continue
        part[c] = r[c]
    parts.append(part)

if parts:
    membership_daily = pd.concat(parts, ignore_index=True)
else:
    membership_daily = pd.DataFrame(columns=["date", "quarter", "firm_id", "name"])

returns_all = pd.read_parquet(OUTPUT_RETURNS_ALL)
returns_all["date"] = pd.to_datetime(returns_all["date"], errors="coerce")
returns_all = returns_all[returns_all["date"] <= TARGET_END_DATE].copy()

ret_cols = [c for c in ["date", "firm_id", "ret", "pull_id", "id_type"] if c in returns_all.columns]
returns_in_index = membership_daily.merge(
    returns_all[ret_cols],
    on=["date", "firm_id"],
    how="left",
    suffixes=("", "_ret"),
)

# keep only dates where we actually have a return
returns_in_index = returns_in_index.dropna(subset=["ret"]).copy()

returns_in_index = returns_in_index.sort_values(["date", "firm_id"]).reset_index(drop=True)

returns_in_index.to_parquet(OUTPUT_DAILY_RETURNS_EURO500, index=False)

print("Built in-index daily returns")
print("Saved:", OUTPUT_DAILY_RETURNS_EURO500)
print("Rows:", len(returns_in_index))
print("Companies covered:", returns_in_index["firm_id"].nunique())
print("Date range:", returns_in_index["date"].min(), "to", returns_in_index["date"].max())


KeyboardInterrupt: 

## 7. QA: Datenabdeckung

Dieser Block zeigt nur das Wesentliche:
- kompakte KPI-Zusammenfassung zur Coverage
- Verlauf der Coverage-Quote je Quartal
- aufgeteilte Anzahl `abgedeckt` vs. `fehlend` je Quartal


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from plot_style import COLORS, set_global_plot_style, style_axes, style_legend, style_time_axis
set_global_plot_style()

if "returns_in_index" in locals():
    returns_in_index = returns_in_index.copy()
else:
    if not OUTPUT_DAILY_RETURNS_EURO500.exists():
        raise FileNotFoundError(f"File not found: {OUTPUT_DAILY_RETURNS_EURO500}. Run Step 6 first.")
    returns_in_index = pd.read_parquet(OUTPUT_DAILY_RETURNS_EURO500).copy()

returns_in_index["date"] = pd.to_datetime(returns_in_index["date"], errors="coerce")
returns_in_index["quarter"] = returns_in_index["quarter"].astype("string")

# Erwartete Firmenzahl je Quartal aus dem Universe
universe_q = (
    df.groupby("quarter", as_index=False)
    .agg(n_members=("firm_id", "nunique"))
)
universe_q["quarter"] = universe_q["quarter"].astype("string")

coverage_q = (
    returns_in_index
    .groupby("quarter", as_index=False)
    .agg(
        n_obs=("ret", "size"),
        n_companies=("firm_id", "nunique"),
        n_dates=("date", "nunique"),
    )
    .merge(universe_q, on="quarter", how="right")
)

coverage_q["n_obs"] = coverage_q["n_obs"].fillna(0).astype(int)
coverage_q["n_companies"] = coverage_q["n_companies"].fillna(0).astype(int)
coverage_q["n_dates"] = coverage_q["n_dates"].fillna(0).astype(int)
coverage_q["n_members"] = coverage_q["n_members"].fillna(0).astype(int)

coverage_q["missing_companies"] = (coverage_q["n_members"] - coverage_q["n_companies"]).clip(lower=0)
coverage_q["coverage_pct"] = np.where(
    coverage_q["n_members"] > 0,
    100.0 * coverage_q["n_companies"] / coverage_q["n_members"],
    np.nan,
)
coverage_q["avg_obs_per_company"] = np.where(
    coverage_q["n_companies"] > 0,
    coverage_q["n_obs"] / coverage_q["n_companies"],
    np.nan,
)
coverage_q["quarter_end"] = pd.PeriodIndex(coverage_q["quarter"].astype(str), freq="Q").to_timestamp(how="end")
QA_END_QUARTER = pd.Period("2025Q4", freq="Q")
coverage_q["quarter_period"] = pd.PeriodIndex(coverage_q["quarter"].astype(str), freq="Q")
coverage_q = coverage_q[coverage_q["quarter_period"] <= QA_END_QUARTER].copy()
coverage_q = coverage_q[coverage_q["quarter_end"] >= pd.Timestamp("1999-01-01")].copy()
coverage_q = coverage_q.sort_values("quarter_end").reset_index(drop=True)

coverage_q["quarter_label_pretty"] = (
    "Q"
    + coverage_q["quarter_period"].dt.quarter.astype(str)
    + " "
    + coverage_q["quarter_period"].dt.year.astype(str)
)

if coverage_q.empty:
    raise ValueError("coverage_q is empty. Check returns_in_index construction and universe input.")

worst_idx = coverage_q["coverage_pct"].idxmin()
worst = coverage_q.loc[worst_idx]
latest = coverage_q.iloc[-1]
mean_cov = float(coverage_q["coverage_pct"].mean())

kpi = pd.DataFrame(
    {
        "metric": [
            "Quarters analyzed",
            "Mean coverage (%)",
            "Median coverage (%)",
            "Quarters < 95% coverage",
            "Worst quarter (coverage %)",
            "Latest quarter (coverage %)",
            "Latest missing companies",
        ],
        "value": [
            int(len(coverage_q)),
            round(float(coverage_q["coverage_pct"].mean()), 2),
            round(float(coverage_q["coverage_pct"].median()), 2),
            int((coverage_q["coverage_pct"] < 95).sum()),
            f"{worst['quarter']} ({worst['coverage_pct']:.2f}%)",
            f"{latest['quarter']} ({latest['coverage_pct']:.2f}%)",
            int(latest["missing_companies"]),
        ],
    }
)
print(kpi.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(14, 9),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1.2]},
)

# Plot 1: Coverage-Quote
ax1.plot(
    coverage_q["quarter_end"],
    coverage_q["coverage_pct"],
    color=COLORS["primary"],
    linewidth=0.9,
    marker="o",
    markersize=2.8,
    label="Coverage (%)",
)
ax1.axhline(mean_cov, color=COLORS["secondary"], linewidth=0.9, linestyle="--", label=f"Mean coverage ({mean_cov:.2f}%)")
ax1.fill_between(
    coverage_q["quarter_end"],
    coverage_q["coverage_pct"],
    97,
    where=coverage_q["coverage_pct"] < 97,
    color=COLORS["accent"],
    alpha=0.15,
)
ax1.scatter([worst["quarter_end"]], [worst["coverage_pct"]], color=COLORS["accent"], s=40, zorder=5)
ax1.annotate(
    f"Worst: {worst['quarter']} ({worst['coverage_pct']:.1f}%)",
    xy=(worst["quarter_end"], worst["coverage_pct"]),
    xytext=(10, -14),
    textcoords="offset points",
    color=COLORS["accent"],
    fontsize=9,
)
ax1.set_title("Daily Return Euro 500: Data Coverage per Quarter", fontsize=13, pad=10)
ax1.set_ylabel("Coverage (%)")
ax1.set_ylim(97, 100.1)
style_axes(ax1, grid_axis="y", label_x=-0.04, label_y=0.5, label_pad=1)
style_legend(ax1, loc="lower right", frameon=False)

# Plot 2: Abgedeckte vs fehlende Firmen
ax2.bar(
    coverage_q["quarter_end"],
    coverage_q["n_companies"],
    width=75,
    color=COLORS["primary"],
    alpha=0.85,
    label="Companies with returns",
)
ax2.bar(
    coverage_q["quarter_end"],
    coverage_q["missing_companies"],
    width=75,
    bottom=coverage_q["n_companies"],
    color=COLORS["accent"],
    alpha=0.75,
    label="Missing companies",
)
ax2.plot(
    coverage_q["quarter_end"],
    coverage_q["n_members"],
    color=COLORS["neutral"],
    linewidth=0.9,
    linestyle="--",
    label="Expected members",
)
ax2.set_title("Daily Return Euro 500: Covered vs Missing Companies", fontsize=13, pad=10)
ax2.set_ylabel("Number of companies")
ax2.set_ylim(480, 500)
ax2.yaxis.set_major_locator(MultipleLocator(5))
style_axes(ax2, grid_axis="y", label_x=-0.04, label_y=0.5, label_pad=1)
style_legend(ax2, loc="lower right", frameon=True)

tick_idx = coverage_q.index[
    (coverage_q["quarter_period"].dt.year >= 2000)
    & (coverage_q["quarter_period"].dt.quarter == 1)
    & (((coverage_q["quarter_period"].dt.year - 2000) % 2) == 0)
].tolist()
if not tick_idx:
    n_q = len(coverage_q)
    step_q = max(1, n_q // 14)
    tick_idx = list(range(0, n_q, step_q))

ax2.set_xticks(coverage_q["quarter_end"].iloc[tick_idx])
ax2.set_xticklabels(coverage_q["quarter_label_pretty"].iloc[tick_idx])
ax2.set_xlabel("Quarter")

step7_plot_path = TABLE_DIR / "lseg_dailyreturns_step7_qa_coverage.png"
plt.tight_layout()
plt.savefig(step7_plot_path, dpi=200, bbox_inches="tight")
plt.show()


## 8. Top 15 Missing-Unternehmen (meiste Quartale im Index)

Diese Zelle zeigt die 15 fehlenden Unternehmen mit den meisten Quartalen im EURO500-Universum als Tabelle.
Zusatzspalten: `sector` sowie `index_period` (von `first_quarter` bis `last_quarter`).



In [ ]:
# Post-Processing only: keine LSEG-Session, kein neuer Datenpull
# NOTE: No additional files are written in this section.

missing_in = OUTPUT_MISSING
euro_in = EURO500_PATH

if not euro_in.exists():
    raise FileNotFoundError(f"euro500 file not found: {euro_in}")

if "missing_df" in locals():
    missing = missing_df.copy()
elif missing_in.exists():
    missing = pd.read_parquet(missing_in).copy()
else:
    missing = pd.DataFrame(columns=["firm_id", "company_name", "ISIN", "RIC", "RIC_current"])
    print("No in-memory missing_df and no missing file found; using empty missing list.")

euro = pd.read_parquet(euro_in).copy()

# Normalize key columns
for c in ["firm_id", "ISIN", "RIC", "RIC_current", "name", "quarter", "date", "trbc_sector", "Sector"]:
    if c in euro.columns:
        euro[c] = euro[c].astype("string").str.strip()
        euro.loc[euro[c] == "", c] = pd.NA

if "firm_id" not in euro.columns:
    raise ValueError("Column 'firm_id' missing in euro500 input.")

euro["firm_id"] = euro["firm_id"].astype("string").str.strip()
euro.loc[euro["firm_id"] == "", "firm_id"] = pd.NA

if "date" in euro.columns:
    euro["date"] = pd.to_datetime(euro["date"], errors="coerce")

# Coverage + period per firm
agg_dict = {
    "n_quarters_in_index": ("quarter", "nunique"),
    "first_quarter": ("quarter", "min"),
    "last_quarter": ("quarter", "max"),
}
if "date" in euro.columns:
    agg_dict["first_date"] = ("date", "min")
    agg_dict["last_date"] = ("date", "max")

firm_period = (
    euro[[c for c in ["firm_id", "quarter", "date"] if c in euro.columns]]
    .dropna(subset=["firm_id", "quarter"])
    .drop_duplicates()
    .groupby("firm_id", as_index=False)
    .agg(**agg_dict)
)

firm_period["index_period"] = (
    firm_period["first_quarter"].astype("string").fillna("")
    + " to "
    + firm_period["last_quarter"].astype("string").fillna("")
)

# Metadata per firm (including sector)
sector_col = "trbc_sector" if "trbc_sector" in euro.columns else ("Sector" if "Sector" in euro.columns else None)
meta_cols = ["firm_id", "name", "RIC", "RIC_current", "ISIN"]
if sector_col:
    meta_cols.append(sector_col)

meta = (
    euro[meta_cols]
    .dropna(subset=["firm_id"])
    .drop_duplicates(subset=["firm_id"], keep="first")
)

if sector_col and sector_col != "sector":
    meta = meta.rename(columns={sector_col: "sector"})
elif "sector" not in meta.columns:
    meta["sector"] = pd.NA

missing_enriched = (
    missing.merge(meta, on="firm_id", how="left", suffixes=("", "_meta"))
    .merge(firm_period, on="firm_id", how="left")
)

# Fill missing id fields from meta fallback
for col in ["firm_id", "ISIN", "RIC", "RIC_current"]:
    mcol = f"{col}_meta"
    if mcol in missing_enriched.columns:
        if col in missing_enriched.columns:
            missing_enriched[col] = missing_enriched[col].fillna(missing_enriched[mcol])
        else:
            missing_enriched[col] = missing_enriched[mcol]

if "name_meta" in missing_enriched.columns:
    name_fallback = missing_enriched["name_meta"].astype("string")
elif "name" in missing_enriched.columns:
    name_fallback = missing_enriched["name"].astype("string")
else:
    name_fallback = pd.Series(pd.NA, index=missing_enriched.index, dtype="string")

if "company_name" in missing_enriched.columns:
    missing_enriched["company_name"] = missing_enriched["company_name"].astype("string").fillna(name_fallback)
else:
    missing_enriched["company_name"] = name_fallback

missing_enriched["n_quarters_in_index"] = missing_enriched["n_quarters_in_index"].fillna(0).astype(int)

# RIC: prefer RIC_current, fallback RIC
ric_series = pd.Series(pd.NA, index=missing_enriched.index, dtype="string")
if "RIC_current" in missing_enriched.columns:
    ric_series = missing_enriched["RIC_current"].astype("string")
if "RIC" in missing_enriched.columns:
    ric_series = ric_series.fillna(missing_enriched["RIC"].astype("string"))

result = pd.DataFrame(
    {
        "name": missing_enriched["company_name"].astype("string"),
        "firmid": missing_enriched["firm_id"].astype("string") if "firm_id" in missing_enriched.columns else pd.Series(pd.NA, index=missing_enriched.index, dtype="string"),
        "ric": ric_series,
        "isin": missing_enriched["ISIN"].astype("string") if "ISIN" in missing_enriched.columns else pd.Series(pd.NA, index=missing_enriched.index, dtype="string"),
        "sector": missing_enriched["sector"].astype("string") if "sector" in missing_enriched.columns else pd.Series(pd.NA, index=missing_enriched.index, dtype="string"),
        "n_quarters_in_index": missing_enriched["n_quarters_in_index"],
        "first_quarter": missing_enriched["first_quarter"].astype("string") if "first_quarter" in missing_enriched.columns else pd.Series(pd.NA, index=missing_enriched.index, dtype="string"),
        "last_quarter": missing_enriched["last_quarter"].astype("string") if "last_quarter" in missing_enriched.columns else pd.Series(pd.NA, index=missing_enriched.index, dtype="string"),
        "index_period": missing_enriched["index_period"].astype("string") if "index_period" in missing_enriched.columns else pd.Series(pd.NA, index=missing_enriched.index, dtype="string"),
    }
)

result = (
    result
    .sort_values(["n_quarters_in_index", "name"], ascending=[False, True], na_position="last")
    .drop_duplicates(subset=["firmid"], keep="first")
    .head(15)
    .reset_index(drop=True)
)

result = result[["name", "firmid", "ric", "isin", "sector", "n_quarters_in_index", "first_quarter", "last_quarter", "index_period"]]

print(f"Missing firms total: {missing['firm_id'].nunique() if 'firm_id' in missing.columns else len(missing)}")
print("Top 15 missing companies by quarters in index:")

try:
    display(result)
except NameError:
    print(result.to_string(index=False))

